# 01 Exploratory Data Analysis (EDA) - Pearls AQI Predictor

**Objective:** Explore air quality distributions, pollutant correlations, seasonality, and weather interactions across the 5 Pakistani cities: **Karachi, Lahore, Islamabad, Peshawar, Quetta**.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pearls_aqi.data.weather_provider import OpenMeteoWeatherProvider
from pearls_aqi.data.air_quality_provider import OpenMeteoAirQualityProvider
from pearls_aqi.data.cleaning import merge_and_clean_city_data
from pearls_aqi.settings import settings

print('Setup complete.')

## 1. Data Collection for January 2025

In [ ]:
cities = settings.load_cities_config()['cities']
weather_provider = OpenMeteoWeatherProvider()
aq_provider = OpenMeteoAirQualityProvider()

dfs = []
for city in cities:
    if city.get('enabled', True):
        w = weather_provider.fetch_historical_weather(city['latitude'], city['longitude'], '2025-01-01', '2025-01-31')
        a = aq_provider.fetch_historical_air_quality(city['latitude'], city['longitude'], '2025-01-01', '2025-01-31')
        m = merge_and_clean_city_data(w, a, city['slug'], city['latitude'], city['longitude'])
        m['city_name'] = city['name']
        dfs.append(m)

df_all = pd.concat(dfs, ignore_index=True)
df_all.head()

## 2. Summary Statistics by City

In [ ]:
df_all.groupby('city_name')['aqi'].describe().round(1)

## 3. Visualization: US AQI Boxplot Distribution

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=df_all, x='city_name', y='aqi', palette='Set2')
plt.title('US AQI Distribution Across Cities (Jan 2025)')
plt.show()